**tạo bảng tracking**

In [2]:
%%sql
create table if not EXISTS dbo.ingestion_log (
    file_name  STRING ,
    status STRING,
    load_time TIMESTAMP
) using DELTA


StatementMeta(, 4574e43c-2adf-46af-9eb7-ea8c7e0fe656, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

**list file trong folder**

In [4]:
files = mssparkutils.fs.ls('Files/taxi_tripdata_raw/')
files_name = [f.name for f in files]

StatementMeta(, 4574e43c-2adf-46af-9eb7-ea8c7e0fe656, 6, Finished, Available, Finished, False)

**list file đã load**

In [ ]:
loaded_files_df = spark.read.table('dbo.ingestion_log')\
                            .filter("status = 'loaded'")\
                            .select('file_name')
loaded_files = [row.file_name for row in loaded_files_df.collect()]

**list file chưa load**

In [ ]:
new_files = [f for f in files_name if f not in loaded_files]

**load file mới vào bảng bronze**

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [12]:
if new_files :
        #tạo list đường dẫn đầy đủ
    path = [f"Files/taxi_tripdata_raw/{file}" for file in new_files]
    #đọc tất cả file mới 
    df = spark.read.parquet(*path)\
                    .withColumn('source_file', F.input_file_name())\
                    .withColumn('ingestion_time', F.current_timestamp())
    #ghi file thành bảng
    df_write = df.write.mode('append').format('delta').saveAsTable('bronze.yellow_tripdata')
    #ghi log vào bảng ingestion_log
    log_data = [(f,'loaded') for f in new_files]   
    log_df = spark.createDataFrame(log_data, ['file_name','status'])\
                  .withColumn('load_time',F.current_timestamp())
    log_df.write.mode('append').format('delta').saveAsTable('dbo.ingestion_log')    

StatementMeta(, 4574e43c-2adf-46af-9eb7-ea8c7e0fe656, 14, Finished, Available, Finished, False)